# Ingest — Viva Insights consumption export

Reads the Consumption Dashboard export from `Files/landing/viva/` and merges it into two Delta
tables: `viva_credits_weekly` and `viva_spending_policy`.

**Why this notebook exists.** The Viva export only reaches back 6 months and there is no API. Run
this weekly and the Delta table accumulates history indefinitely; skip it for six months and that
history is gone for good.

**Idempotent.** Merges on the natural key, so re-running last week's file updates rather than
duplicates. Safe to re-run, safe to backfill out of order.

**Handles both export shapes.** The identified export carries `UserPrincipalName` + `EntraId`;
the de-identified one carries `PersonId` + `PeopleHistoricalId`. Either is normalised to the same
contract, with `person_id` holding the real UPN when identified and the hashed id when not.

In [ ]:
LANDING = "Files/landing/viva"
TBL_METRICS = "viva_credits_weekly"
TBL_POLICY = "viva_spending_policy"

# Set True to archive each file to landing/viva/_processed after a successful merge.
ARCHIVE_AFTER_LOAD = False

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import re


def norm(name):
    """Compare column names ignoring case, spaces, dashes and underscores.

    Export headers drift - 'Session count' vs 'SessionCount' vs 'session_count'.
    Matching on a normalised key means a cosmetic rename upstream does not break
    the pipeline silently.
    """
    return re.sub(r"[ _\-]", "", name).lower()


def pick(df, *aliases):
    """Return the actual column name matching any alias, or None."""
    lookup = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


def col_or_null(df, name, cast="string"):
    """The column if present, otherwise a typed null - so downstream can
    reference both export shapes' columns unconditionally."""
    return F.col(name).cast(cast) if name else F.lit(None).cast(cast)

## Metrics

In [ ]:
raw = (spark.read
       .option("header", True)
       .option("inferSchema", False)
       .csv(f"{LANDING}/PersonServiceCreditsMetrics*.csv"))

print(f"read {raw.count():,} rows")
print("columns:", raw.columns)

c_upn = pick(raw, "UserPrincipalName", "upn")
c_pid = pick(raw, "PersonId")
c_phid = pick(raw, "PeopleHistoricalId")
c_entra = pick(raw, "EntraId", "ObjectId", "AadObjectId")

shape = "identified" if c_upn else "de-identified"
print(f"export shape: {shape}")

if not (c_upn or c_pid):
    raise ValueError(
        "No person key found. Expected UserPrincipalName (identified export) "
        f"or PersonId (de-identified). Columns present: {raw.columns}")

In [ ]:
# person_id is the join key everywhere downstream: the real UPN when the export
# is identified, the hashed id when it is not.
metrics = raw.select(
    F.coalesce(col_or_null(raw, c_pid), col_or_null(raw, c_upn)).alias("person_id"),
    col_or_null(raw, c_upn).alias("user_principal_name"),
    col_or_null(raw, c_entra).alias("entra_id"),
    col_or_null(raw, c_phid).alias("people_historical_id"),
    F.col(pick(raw, "ServiceId")).cast("string").alias("service_id"),
    F.col(pick(raw, "ServiceName")).cast("string").alias("service_name"),
    F.col(pick(raw, "SpendingPolicyId")).cast("string").alias("spending_policy_id"),
    F.to_date(F.col(pick(raw, "MetricDate"))).alias("metric_date"),
    F.col(pick(raw, "Session count")).cast("int").alias("session_count"),
    F.col(pick(raw, "Spending policy limit")).cast("long").alias("spending_policy_limit"),
    F.col(pick(raw, "Total Copilot Credits used")).cast("double").alias("credits_used"),
    F.col(pick(raw, "User limit")).cast("long").alias("user_limit"),
).withColumn("_loaded_at", F.current_timestamp())

weeks = metrics.select("metric_date").distinct().count()
print(f"{metrics.count():,} rows across {weeks} weeks, "
      f"{metrics.select('person_id').distinct().count():,} people")
metrics.select(F.min("metric_date").alias("from"),
               F.max("metric_date").alias("to")).show()

In [ ]:
# A person can hold rows under more than one policy in the same week - usage
# outside any policy window is recorded against the all-zero GUID. The policy is
# therefore part of the key, or a merge would collapse two legitimate rows into
# one and lose credits.
KEY = ["person_id", "service_id", "spending_policy_id", "metric_date"]

if spark.catalog.tableExists(TBL_METRICS):
    before = spark.table(TBL_METRICS).count()
    tgt = DeltaTable.forName(spark, TBL_METRICS)
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in KEY)
    (tgt.alias("t")
        .merge(metrics.alias("s"), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    after = spark.table(TBL_METRICS).count()
    print(f"{TBL_METRICS}: {before:,} -> {after:,}  (+{after - before:,} new)")
else:
    metrics.write.format("delta").saveAsTable(TBL_METRICS)
    print(f"{TBL_METRICS}: created with {metrics.count():,} rows")

## Spending policies

Small and slow-moving, so this one is a full replace rather than a merge.

In [ ]:
pol_raw = (spark.read
           .option("header", True)
           .csv(f"{LANDING}/SpendingPolicyMetadata*.csv"))

policy = pol_raw.select(
    F.col(pick(pol_raw, "SpendingPolicyId")).cast("string").alias("spending_policy_id"),
    # The all-zero GUID row has no name - it marks usage outside any policy.
    F.when(F.col(pick(pol_raw, "Name")).isNull() | (F.col(pick(pol_raw, "Name")) == ""),
           F.lit("(Unassigned)"))
     .otherwise(F.col(pick(pol_raw, "Name"))).alias("name"),
    F.coalesce(F.col(pick(pol_raw, "PlanLimit")).cast("long"), F.lit(0)).alias("plan_limit"),
    F.coalesce(F.col(pick(pol_raw, "UserLimit")).cast("long"), F.lit(0)).alias("user_limit"),
    F.col(pick(pol_raw, "IncludedServices")).cast("string").alias("included_services"),
).withColumn("_loaded_at", F.current_timestamp())

policy.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(TBL_POLICY)
print(f"{TBL_POLICY}: {policy.count()} policies")
policy.show(truncate=False)

## Check

Accumulated history is the whole point of this path, so it is worth looking at every run.

In [ ]:
spark.sql(f"""
    SELECT  MIN(metric_date)                    AS earliest_week,
            MAX(metric_date)                    AS latest_week,
            COUNT(DISTINCT metric_date)         AS weeks_held,
            COUNT(DISTINCT person_id)           AS people,
            ROUND(SUM(credits_used))            AS total_credits
    FROM    {TBL_METRICS}
""").show(truncate=False)

# Weeks_held above the export's own 6-month window means history is accruing -
# which is the reason to be on this path at all.
spark.sql(f"""
    SELECT  metric_date, COUNT(*) AS rows, ROUND(SUM(credits_used)) AS credits
    FROM    {TBL_METRICS}
    GROUP BY metric_date
    ORDER BY metric_date DESC
    LIMIT   12
""").show(truncate=False)

In [ ]:
if ARCHIVE_AFTER_LOAD:
    import datetime
    import notebookutils

    stamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    dest = f"{LANDING}/_processed/{stamp}"
    notebookutils.fs.mkdirs(dest)
    for f in notebookutils.fs.ls(LANDING):
        if f.name.lower().endswith(".csv"):
            notebookutils.fs.mv(f.path, f"{dest}/{f.name}")
            print(f"archived {f.name}")
else:
    print("ARCHIVE_AFTER_LOAD is False - files left in place.")
    print("Re-running is harmless: the merge updates rather than duplicates.")